Sentiment Analysis Model Training

Goal: Train a model to perform a sentiment analysis on customer call transcripts.

Datasets:

call_transcripts.csv  
Steps:

Load & explore data
Feature engineering & preprocessing
Train/test split
Train model
Evaluate performance

In [131]:
# %pip uninstall -y boto3
# %pip uninstall -y s3fs
# %pip uninstall -y sagemaker
%pip install "transformers==4.36.2" "accelerate==0.24.1"
%pip install datasets
%pip install accelerate

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
from typing import Any, NoReturn, Literal
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments)
from datasets import Dataset, DatasetDict


import warnings
warnings.filterwarnings(action='ignore')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [132]:
# Load dataset
file_path = "~/Downloads/call_transcripts.csv"
df = pd.read_csv(filepath_or_buffer=file_path, sep=",", engine="python", encoding="utf-8", encoding_errors="strict")

In [133]:
# Explore data
print(f'Internet data: {df.shape[0]} rows, {df.shape[1]} columns')

Internet data: 2500 rows, 13 columns


In [134]:
print('--- Customer Data ---')
df.head(n=10)

--- Customer Data ---


,call_id,customer_id,call_date,call_time,agent_id,agent_name,primary_scenario,call_transcript,overall_rating,call_successful,customer_monthly_spend,customer_service_count,customer_issue_history
0,CALL_000001,C00077940,2025-04-26,12:08,agent_005,Lisa Wang,payment_assistance,**Call Transcript**\n\n**Date:** 2025-04-26\n*...,5,True,173,2,18
1,CALL_000002,C00050897,2025-01-31,14:51,agent_007,Jennifer Davis,billing_inquiry,**Call Transcript**\n\n**Date:** 2025-01-31\n*...,5,True,182,2,0
2,CALL_000003,C00062906,2025-08-26,16:46,agent_008,Robert Kim,contract_renewal,**TriLink Telecom Customer Service Call Transc...,5,False,176,2,0
3,CALL_000004,C00077227,2025-06-13,17:59,agent_006,Michael Brown,technical_support,**Call Transcript: TriLink Telecom Customer Se...,7,True,100,1,3
4,CALL_000005,C00012668,2025-08-13,11:40,agent_004,James Thompson,cross_sell_security,**TriLink Telecom Customer Service Call Transc...,4,False,36,1,3
5,CALL_000006,C00005383,2025-06-23,11:55,agent_010,Carlos Martinez,complaint_resolution,**TriLink Telecom Customer Service Call Transc...,7,True,394,3,7
6,CALL_000007,C00072547,2025-02-15,08:34,agent_003,Maria Rodriguez,billing_inquiry,**TriLink Telecom Customer Service Call Transc...,6,True,52,1,1
7,CALL_000008,C00084129,2025-04-10,08:21,agent_002,David Chen,technical_support,**Call Transcript**\n\n**Date:** 2025-04-10\n*...,7,True,47,1,0
8,CALL_000009,C00064140,2025-04-04,11:21,agent_006,Michael Brown,technical_support,**TriLink Telecom Customer Service Transcript*...,4,False,152,1,0
9,CALL_000010,C00002870,2025-07-17,11:17,agent_009,Ashley Johnson,cross_sell_internet,**TriLink Telecom Customer Service Call Transc...,7,True,260,2,9


In [135]:
print('--- Customer Data ---')
df.sample(n=10, random_state=3)

--- Customer Data ---


,call_id,customer_id,call_date,call_time,agent_id,agent_name,primary_scenario,call_transcript,overall_rating,call_successful,customer_monthly_spend,customer_service_count,customer_issue_history
1015,CALL_001016,C00096341,2025-07-15,14:48,agent_003,Maria Rodriguez,billing_inquiry,**TriLink Telecom Customer Service Call Transc...,4,False,63,1,8
972,CALL_000973,C00090738,2025-06-02,11:45,agent_002,David Chen,technical_support,**Call Transcript: TriLink Telecom Customer Se...,7,True,185,3,9
2178,CALL_002179,C00013093,2025-01-19,15:08,agent_006,Michael Brown,technical_support,**(Call begins with a standard automated greet...,5,False,53,1,6
2381,CALL_002382,C00022969,2025-08-08,15:26,agent_009,Ashley Johnson,cross_sell_mobile,**Call Transcript**\n\n**Date:** 2025-08-08\n*...,6,True,65,1,10
1797,CALL_001798,C00021010,2025-04-28,15:32,agent_007,Jennifer Davis,upsell_security_devices,**TriLink Telecom Customer Service Call Transc...,5,True,270,3,13
319,CALL_000320,C00055616,2025-08-18,15:11,agent_002,David Chen,payment_assistance,**Call Transcript**\n\n**Date:** 2025-08-18\n*...,5,True,108,2,19
1458,CALL_001459,C00069954,2025-06-01,10:45,agent_004,James Thompson,service_cancellation,**TriLink Telecom Customer Service Call Transc...,6,False,117,1,10
1363,CALL_001364,C00049079,2025-05-26,08:34,agent_009,Ashley Johnson,service_cancellation,**Date:** 2025-05-26\n**Agent:** Ashley Johnso...,7,True,377,2,2
1057,CALL_001058,C00047482,2025-01-03,08:44,agent_003,Maria Rodriguez,billing_inquiry,**Call Transcript: TriLink Telecom Customer Se...,6,True,334,3,1
334,CALL_000335,C00043068,2025-05-31,12:18,agent_006,Michael Brown,technical_support,**TriLink Telecom Customer Service Call Transc...,7,True,211,2,1


In [136]:
print('--- Customer Data Info ---')
df.info()

--- Customer Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   call_id                 2500 non-null   object
 1   customer_id             2500 non-null   object
 2   call_date               2500 non-null   object
 3   call_time               2500 non-null   object
 4   agent_id                2500 non-null   object
 5   agent_name              2500 non-null   object
 6   primary_scenario        2500 non-null   object
 7   call_transcript         2500 non-null   object
 8   overall_rating          2500 non-null   int64 
 9   call_successful         2500 non-null   bool  
 10  customer_monthly_spend  2500 non-null   int64 
 11  customer_service_count  2500 non-null   int64 
 12  customer_issue_history  2500 non-null   int64 
dtypes: bool(1), int64(4), object(8)
memory usage: 236.9+ KB


In [137]:
print('--- Customer Data Statistics---')
df.describe()

--- Customer Data Statistics---


,overall_rating,customer_monthly_spend,customer_service_count,customer_issue_history
count,2500.000000,2500.000000,2500.000000,2500.000000
mean,5.834000,208.087200,2.028800,4.152800
std,0.964373,128.998527,0.767467,4.308512
min,3.000000,12.000000,1.000000,0.000000
25%,5.000000,99.000000,1.000000,1.000000
50%,6.000000,187.500000,2.000000,3.000000
75%,6.000000,305.250000,3.000000,6.000000
max,8.000000,646.000000,3.000000,27.000000


In [138]:
print('--- Customer Data Types---')
df.select_dtypes(include="object").nunique()

--- Customer Data Types---


call_id             2500
customer_id         2000
call_date            244
call_time            592
agent_id              10
agent_name            10
primary_scenario      14
call_transcript     2500
dtype: int64

In [139]:
# Check for null values
print('--- Customer Data Null ---')
print(df.isnull().sum())
print('--- Customer Data Sum Non Null ---')
print(df.notnull().value_counts().sum())

--- Customer Data Null ---
call_id                   0
customer_id               0
call_date                 0
call_time                 0
agent_id                  0
agent_name                0
primary_scenario          0
call_transcript           0
overall_rating            0
call_successful           0
customer_monthly_spend    0
customer_service_count    0
customer_issue_history    0
dtype: int64
--- Customer Data Sum Non Null ---
2500


In [140]:
import re

def sanitize_transcript(text) -> str| None | Literal['']:
    if pd.isna(text):
        return ""
    # Remove everything before the first '---'
    if "---" in text:
        text = text.split("---", 1)[1]
    # Remove **bold markdown**
    text = re.sub(pattern=r"\*\*(.*?)\*\*", repl="", string=text)
    # Remove (parentheses content)
    text = re.sub(pattern=r"\([^)]*\)", repl="", string=text)
    # Remove [bracket content]
    text = re.sub(pattern=r"\[[^\] ]*\]", repl="", string=text)
    # Normalize whitespace
    text = re.sub(pattern=r"\s+", repl=" ", string=text).strip()

    return text

df["clean_transcript"] = df["call_transcript"].apply(func=sanitize_transcript)
df["clean_transcript"].iloc[0]

"Thank you for calling TriLink Telecom, my name is Lisa, how can I help you today? Yeah, hi Lisa. Um, I need some help with my bill. My latest one just came in, and it's… it's really high. I think I need to set up a payment arrangement or something. Okay, no problem, I can certainly check into that for you. To start, could you please verify your full name and the phone number associated with your account? Sure. It's Alex Chen, and the number is 555-876-4321. Thank you, Alex. And just for security, could you confirm your date of birth, please? March 15th, 1996. Perfect, thank you. Just a moment while I pull up your account... Okay, I see your account here, C00077940. Your current services are the Standard_100 Internet plan at $74 a month, and the Limited_10GB mobile plan with two lines for $99 a month. Is that all correct? Yeah, that's right. And my bill's $173, right? That's correct, your total due for this billing cycle is $173.00, and the due date is May 1st. Right. Look, I'm just… I

In [141]:
def engineer_features(df) -> Any:
    df = df.copy()
    df["word_count"] = df["call_transcript"].str.split().str.len()
    df["avg_word_length"] = (
        df["call_transcript"]
        .str.split()
        .apply(lambda words: sum(len(w) for w in words) / len(words) if words else 0)
    )
    df["num_exclamation_marks"] = df["call_transcript"].str.count("!")
    return df

df = engineer_features(df=df)
df.head()


,call_id,customer_id,call_date,call_time,agent_id,agent_name,primary_scenario,call_transcript,overall_rating,call_successful,customer_monthly_spend,customer_service_count,customer_issue_history,clean_transcript,word_count,avg_word_length,num_exclamation_marks
0,CALL_000001,C00077940,2025-04-26,12:08,agent_005,Lisa Wang,payment_assistance,**Call Transcript**\n\n**Date:** 2025-04-26\n*...,5,True,173,2,18,"Thank you for calling TriLink Telecom, my name...",974,4.946612,2
1,CALL_000002,C00050897,2025-01-31,14:51,agent_007,Jennifer Davis,billing_inquiry,**Call Transcript**\n\n**Date:** 2025-01-31\n*...,5,True,182,2,0,"Thank you for calling TriLink Telecom, my name...",1653,5.149425,2
2,CALL_000003,C00062906,2025-08-26,16:46,agent_008,Robert Kim,contract_renewal,**TriLink Telecom Customer Service Call Transc...,5,False,176,2,0,Thank you for calling TriLink Telecom. Please ...,1343,5.079672,1
3,CALL_000004,C00077227,2025-06-13,17:59,agent_006,Michael Brown,technical_support,**Call Transcript: TriLink Telecom Customer Se...,7,True,100,1,3,2025-06-13 10:17 AM EST Michael Brown Ms. Davi...,1191,5.168766,2
4,CALL_000005,C00012668,2025-08-13,11:40,agent_004,James Thompson,cross_sell_security,**TriLink Telecom Customer Service Call Transc...,4,False,36,1,3,"Thank you for calling TriLink Telecom, my name...",1292,4.932663,3


In [142]:

###### Transformer Based Approach to Training Model #####
df = df.dropna(subset=["call_transcript", "overall_rating"])
df.head()

,call_id,customer_id,call_date,call_time,agent_id,agent_name,primary_scenario,call_transcript,overall_rating,call_successful,customer_monthly_spend,customer_service_count,customer_issue_history,clean_transcript,word_count,avg_word_length,num_exclamation_marks
0,CALL_000001,C00077940,2025-04-26,12:08,agent_005,Lisa Wang,payment_assistance,**Call Transcript**\n\n**Date:** 2025-04-26\n*...,5,True,173,2,18,"Thank you for calling TriLink Telecom, my name...",974,4.946612,2
1,CALL_000002,C00050897,2025-01-31,14:51,agent_007,Jennifer Davis,billing_inquiry,**Call Transcript**\n\n**Date:** 2025-01-31\n*...,5,True,182,2,0,"Thank you for calling TriLink Telecom, my name...",1653,5.149425,2
2,CALL_000003,C00062906,2025-08-26,16:46,agent_008,Robert Kim,contract_renewal,**TriLink Telecom Customer Service Call Transc...,5,False,176,2,0,Thank you for calling TriLink Telecom. Please ...,1343,5.079672,1
3,CALL_000004,C00077227,2025-06-13,17:59,agent_006,Michael Brown,technical_support,**Call Transcript: TriLink Telecom Customer Se...,7,True,100,1,3,2025-06-13 10:17 AM EST Michael Brown Ms. Davi...,1191,5.168766,2
4,CALL_000005,C00012668,2025-08-13,11:40,agent_004,James Thompson,cross_sell_security,**TriLink Telecom Customer Service Call Transc...,4,False,36,1,3,"Thank you for calling TriLink Telecom, my name...",1292,4.932663,3


In [143]:
from sklearn.model_selection import train_test_split
# Normalize before splitting:
df["overall_rating"] -= df["overall_rating"].min()

train_df, test_df = train_test_split(
    df, stratify=df["overall_rating"], test_size=0.2, random_state=42
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})


In [144]:

# Tokenize the dataset
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch) -> Any:
    return tokenizer(
        batch["call_transcript"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized = dataset.map(tokenize_batch, batched=True)
tokenized = tokenized.remove_columns(
    [c for c in tokenized["train"].column_names if c not in ["input_ids", "attention_mask", "overall_rating"]]
)
tokenized = tokenized.rename_column("overall_rating", "labels")
tokenized.set_format("torch")


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [145]:
# Load the model
num_labels = int(df["overall_rating"].nunique())


model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [146]:
# Define Training arguments
training_args = TrainingArguments(
    output_dir="./sentiment_model",
    logging_dir='./logs',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    warmup_steps=500,
    no_cuda=True
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [147]:

# Define compute metrics function
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def compute_metrics(eval_pred) -> dict[str, Any]:
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true=labels, y_pred=preds),
        "f1_macro": f1_score(y_true=labels, y_pred=preds, average="macro"),
    }


def compare_compute_metrics(pred) -> dict[str, Any]:
    labels = pred.label_ids
    preds = np.argmax(a=pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true=labels, y_pred=preds, average='weighted')
    acc = accuracy_score(y_true=labels, y_pred=preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}


In [148]:
# Disable Weights & Biases (W&B)
os.environ["WANDB_DISABLED"] = "true"

In [149]:
print(sorted(df["overall_rating"].unique()))
print(df["overall_rating"].nunique())

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
6


In [150]:
import mlflow
mlflow.autolog(disable=True)

# End and Start a new session
mlflow.end_run()
mlflow.start_run()
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model on the larger subset
trainer.train()
trainer.evaluate()

  0%|          | 0/500 [00:00<?, ?it/s]

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'loss': 1.7466, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.2}
{'loss': 1.5808, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.4}
{'loss': 1.3956, 'learning_rate': 6e-06, 'epoch': 0.6}
{'loss': 1.3445, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.8}
{'loss': 1.4109, 'learning_rate': 1e-05, 'epoch': 1.0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.3637497425079346, 'eval_accuracy': 0.434, 'eval_f1_macro': 0.10088331008833101, 'eval_runtime': 18.4277, 'eval_samples_per_second': 27.133, 'eval_steps_per_second': 3.419, 'epoch': 1.0}
{'loss': 1.3543, 'learning_rate': 1.2e-05, 'epoch': 1.2}
{'loss': 1.3134, 'learning_rate': 1.4e-05, 'epoch': 1.4}
{'loss': 1.2304, 'learning_rate': 1.6000000000000003e-05, 'epoch': 1.6}
{'loss': 1.2669, 'learning_rate': 1.8e-05, 'epoch': 1.8}
{'loss': 1.2922, 'learning_rate': 0.0, 'epoch': 2.0}


  0%|          | 0/63 [00:00<?, ?it/s]

Checkpoint destination directory ./sentiment_model/checkpoint-500 already exists and is non-empty.Saving will proceed but saved results may be invalid.


{'eval_loss': 1.272201657295227, 'eval_accuracy': 0.438, 'eval_f1_macro': 0.25157233455787736, 'eval_runtime': 18.591, 'eval_samples_per_second': 26.895, 'eval_steps_per_second': 3.389, 'epoch': 2.0}
{'train_runtime': 627.6372, 'train_samples_per_second': 6.373, 'train_steps_per_second': 0.797, 'train_loss': 1.393567642211914, 'epoch': 2.0}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.272201657295227,
 'eval_accuracy': 0.438,
 'eval_f1_macro': 0.25157233455787736,
 'eval_runtime': 18.7417,
 'eval_samples_per_second': 26.679,
 'eval_steps_per_second': 3.361,
 'epoch': 2.0}

In [151]:
id2label = {i: str(object=i) for i in range(num_labels)}

def predict_call(row) -> dict[str, Any]:
    text = row["call_transcript"]
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**{k: v for k, v in inputs.items()})
        probs = torch.softmax(input=outputs.logits, dim=-1)[0].cpu().numpy()
        pred_idx = int(np.argmax(a=probs))
        confidence = float(probs[pred_idx])

    return {
        "call_id": row.get("call_id"),
        "customer_id": row.get("customer_id"),
        "primary_scenario": row.get("primary_scenario"),

        "qa_score": None,
        "sentiment": id2label[pred_idx],
        "category": row.get("primary_scenario"),
        "confidence": confidence,
        "frustration_level": None,
        "call_duration_indicator": None,
        "escalation_flag": None,

        "word_count": len(text.split()),
        "avg_word_length": (
            sum(len(w) for w in text.split()) / max(1, len(text.split()))
        ),

        "emotion_scores": {
            "anger": None,
            "sadness": None,
            "frustration": None,
        },

        "billing_dispute_flag": None,
        "outage_history_flag": None,
        "overage_amount_last_cycle": None,

        "agent_experience": None,
        "transfer_count": None,
        "resolution_flag": row.get("call_successful"),
    }


In [152]:
sample = df.iloc[0]
predict_call(row=sample)


{'call_id': 'CALL_000001',
 'customer_id': 'C00077940',
 'primary_scenario': 'payment_assistance',
 'qa_score': None,
 'sentiment': '1',
 'category': 'payment_assistance',
 'confidence': 0.36170682311058044,
 'frustration_level': None,
 'call_duration_indicator': None,
 'escalation_flag': None,
 'word_count': 974,
 'avg_word_length': 4.946611909650924,
 'emotion_scores': {'anger': None, 'sadness': None, 'frustration': None},
 'billing_dispute_flag': None,
 'outage_history_flag': None,
 'overage_amount_last_cycle': None,
 'agent_experience': None,
 'transfer_count': None,
 'resolution_flag': np.True_}

In [155]:

import torch.nn.functional as F
device = torch.device(device="cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_row(row) -> dict[str, any]:
    text = row["call_transcript"]

    # --- Tokenize ---
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    ).to(device)

    # --- Forward pass ---
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)

    # --- Prediction ---
    pred = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred].item()

    # --- Basic text stats ---
    word_count = len(text.split())
    avg_word_length = sum(len(w) for w in text.split()) / max(word_count, 1)
    num_exclamation_marks = text.count("!")

    # --- Frustration heuristic ---
    frustration_keywords = [
        "angry", "upset", "frustrated", "ridiculous", "unacceptable",
        "this is crazy", "i'm tired of", "why do i have to"
    ]
    frustration_level = sum(1 for kw in frustration_keywords if kw in text.lower())

    # --- Emotion scores ---
    emotion_scores = {
        "anger": sum(1 for w in ["angry", "furious", "mad"] if w in text.lower()),
        "sadness": sum(1 for w in ["sad", "disappointed", "unhappy"] if w in text.lower()),
        "frustration": frustration_level
    }

    # --- Call duration indicator ---
    duration = row.get("call_duration_seconds")
    call_duration_indicator = (
        "short" if duration and duration < 180 else
        "medium" if duration and duration < 600 else
        "long" if duration else None
    )

    # --- Escalation flag ---
    escalation_flag = any(
        phrase in text.lower()
        for phrase in [
            "let me speak to a supervisor",
            "i want a manager",
            "this needs escalation",
            "transfer me"
        ]
    )

    # --- Billing / service flags ---
    billing_dispute_flag = any(
        phrase in text.lower()
        for phrase in ["charge", "billing", "refund", "overcharged", "invoice"]
    )

    outage_history_flag = any(
        phrase in text.lower()
        for phrase in ["outage", "service down", "no signal", "network issue"]
    )

    return {
        "call_id": row["call_id"],
        "customer_id": row["customer_id"],
        "primary_scenario": row["primary_scenario"],

        # model outputs
        "sentiment": int(pred),
        "confidence": float(confidence),

        # engineered features
        "word_count": word_count,
        "avg_word_length": avg_word_length,
        "num_exclamation_marks": num_exclamation_marks,
        "frustration_level": frustration_level,
        "emotion_scores": emotion_scores,
        "call_duration_indicator": call_duration_indicator,
        "escalation_flag": escalation_flag,

        # billing / service
        "billing_dispute_flag": billing_dispute_flag,
        "outage_history_flag": outage_history_flag,
        "overage_amount_last_cycle": row.get("overage_amount_last_cycle"),

        # agent behavior
        "agent_experience": row.get("agent_experience"),
        "transfer_count": row.get("transfer_count"),
        "resolution_flag": any(
            phrase in text.lower()
            for phrase in ["resolved", "fixed", "taken care of", "issue closed"]
        ),

        # passthrough fields
        "qa_score": row.get("qa_score"),
        "category": row.get("category")
    }
    
results = df.apply(predict_row, axis=1)
results.iloc[0]

{'call_id': 'CALL_000001',
 'customer_id': 'C00077940',
 'primary_scenario': 'payment_assistance',
 'sentiment': 1,
 'confidence': 0.36170682311058044,
 'word_count': 974,
 'avg_word_length': 4.946611909650924,
 'num_exclamation_marks': 2,
 'frustration_level': 2,
 'emotion_scores': {'anger': 0, 'sadness': 0, 'frustration': 2},
 'call_duration_indicator': None,
 'escalation_flag': True,
 'billing_dispute_flag': True,
 'outage_history_flag': True,
 'overage_amount_last_cycle': None,
 'agent_experience': None,
 'transfer_count': None,
 'resolution_flag': False,
 'qa_score': None,
 'category': None}

In [156]:
pred_df = pd.DataFrame(data=results.tolist())
pred_df.head()

,call_id,customer_id,primary_scenario,sentiment,confidence,word_count,avg_word_length,num_exclamation_marks,frustration_level,emotion_scores,call_duration_indicator,escalation_flag,billing_dispute_flag,outage_history_flag,overage_amount_last_cycle,agent_experience,transfer_count,resolution_flag,qa_score,category
0,CALL_000001,C00077940,payment_assistance,1,0.361707,974,4.946612,2,2,"{'anger': 0, 'sadness': 0, 'frustration': 2}",None,True,True,True,None,None,None,False,None,None
1,CALL_000002,C00050897,billing_inquiry,2,0.359161,1653,5.149425,2,1,"{'anger': 0, 'sadness': 0, 'frustration': 1}",None,True,True,False,None,None,None,False,None,None
2,CALL_000003,C00062906,contract_renewal,3,0.437841,1343,5.079672,1,0,"{'anger': 1, 'sadness': 0, 'frustration': 0}",None,False,True,False,None,None,None,False,None,None
3,CALL_000004,C00077227,technical_support,3,0.532074,1191,5.168766,2,2,"{'anger': 1, 'sadness': 0, 'frustration': 2}",None,False,True,True,None,None,None,True,None,None
4,CALL_000005,C00012668,cross_sell_security,3,0.446409,1292,4.932663,3,2,"{'anger': 0, 'sadness': 0, 'frustration': 2}",None,True,True,True,None,None,None,False,None,None


In [ ]:
%pip install text2emotion
%pip install emoji==1.7.0

import text2emotion as te

import nltk
nltk.download(info_or_id='punkt_tab')

def extract_emotions(text) -> dict[str, int] | Any:
    if not isinstance(text, str) or not text.strip():
        return {"anger":0, "fear":0, "happy":0, "sad":0, "surprise":0}
    return te.get_emotion(input=text)

emotion_df = df["clean_transcript"].apply(extract_emotions).apply(pd.Series)
emotion_df.columns = [f"emotion_{c}" for c in emotion_df.columns]

df = pd.concat(objs=[df, emotion_df], axis=1)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package stopwords to /Users/om/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/om/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/om/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/om/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
%pip install vaderSentiment

from typing import Literal

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def toxicity_score(text) -> float | Literal[0]:
    if not isinstance(text, str):
        return 0
    scores = analyzer.polarity_scores(text=text)
    # Toxicity proxy = negative sentiment intensity
    return scores["neg"]

df["toxicity_score"] = df["clean_transcript"].apply(toxicity_score)



In [ ]:

FRUSTRATION_WORDS = [
    # direct frustration
    "frustrated", "frustrating", "fed up", "annoyed", "upset", "angry",
    "mad", "irritated", "ridiculous", "unacceptable", "insane", "absurd",

    # service issues
    "slow", "slowing down", "cutting out", "dropping", "intermittent",
    "not working", "never works", "always failing", "constant issues",
    "persistent issues", "ongoing issues", "keeps happening",
    "same problem", "nothing helps", "nothing works",

    # billing frustration
    "high bill", "massive bill", "wrong bill", "unexpected charges",
    "overage charges", "late fee", "charged extra", "bill is wrong",
    "bill is too high", "can't afford", "feels like a lot",

    # escalation-level frustration
    "really fed up", "close to switching", "thinking of switching",
    "might cancel", "i'm done", "tired of this", "called multiple times",
    "keeps happening", "nobody told me", "nobody informed me",
    "shouldn't have to", "don't have time for this", "wasn't handled well",

    # emotional intensity
    "nightmare", "hassle", "mess", "terrible", "awful", "stressful",
    "confusing", "overwhelmed", "exhausted", "sick of this",

    # reliability complaints
    "completely unreliable", "keeps cutting out", "nowhere near",
    "video calls drop", "downloads fail", "signal degradation",
    "keeps going out"
]


def frustration_level(text) -> int:
    if not isinstance(text, str):
        return 0
    text_lower = text.lower()
    count = sum(text_lower.count(w) for w in FRUSTRATION_WORDS)
    # scale to 0–10
    return min(10, count * 2)

df["frustration_level"] = df["clean_transcript"].apply(frustration_level)


In [ ]:
def escalation_probability(row) -> float:
    score = 0
    text = row["clean_transcript"].lower()
    if "cancel" in text or "switch providers" in text:
        score += 4
    if row["frustration_level"] >= 6:
        score += 3
    if row["toxicity_score"] > 0.3:
        score += 2
    if "billing" in text or "overage" in text:
        score += 1
    if "outage" in text or "slow" in text:
        score += 1
    return min(1.0, score / 10)

df["escalation_probability"] = df.apply(escalation_probability, axis=1)


In [ ]:
def agent_turns(text, agent_name) -> int:
    if not isinstance(text, str):
        return 0
    prefix = f"{agent_name}:"
    return text.count(prefix)

def customer_turns(text) -> int:
    if not isinstance(text, str):
        return 0
    return text.count("Customer:")

df["agent_turns"] = df.apply(
    lambda row: agent_turns(text=row["clean_transcript"], agent_name=row["agent_name"]),
    axis=1
)
df["customer_turns"] = df["clean_transcript"].apply(customer_turns)


In [ ]:
EMPATHY_PHRASES = [
    "i understand",
    "i completely understand",
    "i hear your frustration",
    "i hear you",
    "i hear your concern",
    "i understand your frustration",
    "i understand how disruptive",
    "i know how frustrating",
    "i see why you're upset",
    "i can see why",
    "i understand why",

    "i apologize",
    "i sincerely apologize",
    "i'm sorry",
    "i truly apologize",
    "i apologize for the inconvenience",
    "i apologize if",

    "i see a high number",
    "i can see your service history",
    "i understand this has been going on",
    "i know you've called",
    "i see this has been a recurring issue",

    "i can definitely help",
    "i can certainly look into",
    "i'll make sure",
    "i'll take care of that",
    "i want to help resolve this",

    "let me check",
    "let me take a look",
    "let me see what i can do",
    "let me review your account",

    "thank you for your patience",
    "thank you for checking",
    "i appreciate you explaining",
    "i appreciate your time",

    "i understand this feels like a lot",
    "i know this unexpected increase is frustrating",
    "i understand you're having a tough time",

    "i know how important reliable service is",
    "i completely understand how that affects your work",
]


def empathy_score(text) -> int:
    text = text.lower()
    return sum(text.count(p.lower()) for p in EMPATHY_PHRASES)

df["agent_empathy_score"] = df["clean_transcript"].apply(empathy_score)


In [ ]:
def billing_dispute_flag(text) -> bool:
    text = text.lower()
    return any(word in text for word in ["bill", "charge", "overage", "credit", "fee"])

def outage_history_flag(text) -> bool:
    text = text.lower()
    return any(word in text for word in ["outage", "slow", "disconnect", "no service"])

df["billing_dispute_flag"] = df["clean_transcript"].apply(billing_dispute_flag)
df["outage_history_flag"] = df["clean_transcript"].apply(outage_history_flag)


In [ ]:
def build_features(df) -> NoReturn:
    df = df.copy()
    # Emotion
    emotion_df = df["clean_transcript"].apply(extract_emotions).apply(pd.Series)
    emotion_df.columns = [f"emotion_{c}" for c in emotion_df.columns]
    df = pd.concat(objs=[df, emotion_df], axis=1)
    # Toxicity
    df["toxicity_score"] = df["clean_transcript"].apply(toxicity_score)
    # Frustration
    df["frustration_level"] = df["clean_transcript"].apply(frustration_level)
    # Escalation
    df["escalation_probability"] = df.apply(escalation_probability, axis=1)
    # Agent behavior
    df["agent_turns"] = df["clean_transcript"].apply(agent_turns)
    df["customer_turns"] = df["clean_transcript"].apply(customer_turns)
    df["agent_talk_ratio"] = df["agent_turns"] / (df["agent_turns"] + df["customer_turns"] + 1e-6)
    df["customer_talk_ratio"] = df["customer_turns"] / (df["agent_turns"] + df["customer_turns"] + 1e-6)
    df["agent_empathy_score"] = df["clean_transcript"].apply(empathy_score)
    # Billing flags
    df["billing_dispute_flag"] = df["clean_transcript"].apply(billing_dispute_flag)
    df["outage_history_flag"] = df["clean_transcript"].apply(outage_history_flag)

    return df


In [ ]:
df_features = build_features(df=df)
df_features.head()

In [ ]:

# s3 = boto3.client(service_name="s3")

# bucket = "retention-engine-bucket"
# key = "data/call_transcripts.csv"

# Download to local notebook environment

# download = s3.download_file(Bucket=bucket, Key=key, Filename="call_transcripts.csv")
# print(download)

# Read directly to memory
# df = pd.read_csv(filepath_or_buffer=download["Body"])